In [ ]:


import os
import json
import random
import math
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

PHASE3_DIR = "/content/drive/MyDrive/PACL_Project/phase3_outputs"
CKPT_DIR = f"{PHASE3_DIR}/checkpoints_byt5base_vanilla_supcon"
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42

def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)
print(f"✅ seed سراسری روی {SEED} تنظیم شد (Python/NumPy/Torch/CUDA)")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_records = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
val_records   = load_jsonl(f"{PHASE3_DIR}/val.jsonl")

print("train:", len(train_records), "| شاعرانِ یکتا:", len(set(r["poet"] for r in train_records)))
print("val:  ", len(val_records),   "| شاعرانِ یکتا:", len(set(r["poet"] for r in val_records)))

assert set(r["poet"] for r in val_records) <= set(r["poet"] for r in train_records), \
    "❌ val شامل شاعرانی است که در train نیستند!"

def clean(v):
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

def build_vocab(field, records):
    vals = {clean(r.get(field)) for r in records}
    vals.discard(None)
    return {v: i for i, v in enumerate(sorted(vals))}

poet_to_id   = build_vocab("poet", train_records)
rhythm_to_id = build_vocab("rhythm", train_records)
school_to_id = build_vocab("school", train_records)
format_to_id = build_vocab("format", train_records)

assert len(poet_to_id) == 150, f"باید ۱۵۰ شاعر باشد، شد {len(poet_to_id)}"
print("تعداد شاعران:", len(poet_to_id), "| وزن‌های یکتا:", len(rhythm_to_id),
      "| مکاتب:", len(school_to_id), "| قالب‌ها:", len(format_to_id))

In [ ]:


import random
from collections import defaultdict

class FullSweepSchoolGroupedSampler:

    def __init__(self, records, poet_key="poet", school_key="school",
                 primary_per_batch=3, secondary_per_batch=1,
                 poems_per_poet=2, seed=None):
        self.records = records
        self.P = primary_per_batch
        self.S = secondary_per_batch
        self.K = poems_per_poet
        self.rng = random.Random(seed)

        self.by_poet = defaultdict(list)
        poet_school = {}
        for idx, r in enumerate(records):
            self.by_poet[r[poet_key]].append(idx)
            poet_school[r[poet_key]] = r[school_key]
        self.poet_school = poet_school

        self.school_to_poets = defaultdict(list)
        for poet, school in poet_school.items():
            self.school_to_poets[school].append(poet)

    def _poet_occurrences(self, poet):
        idxs = self.by_poet[poet][:]
        self.rng.shuffle(idxs)
        chunks = [idxs[i:i + self.K] for i in range(0, len(idxs), self.K)]
        if chunks and len(chunks[-1]) < self.K:
            deficit = self.K - len(chunks[-1])
            chunks[-1] = chunks[-1] + [self.rng.choice(idxs) for _ in range(deficit)]
        return chunks

    def __iter__(self):
        queues = defaultdict(list)
        for school, poets in self.school_to_poets.items():
            occ = []
            for poet in poets:
                for chunk in self._poet_occurrences(poet):
                    occ.append((poet, chunk))
            self.rng.shuffle(occ)
            queues[school] = occ

        while any(queues.values()):
            nonempty = [s for s in queues if queues[s]]
            primary_school = max(nonempty, key=lambda s: len(queues[s]))
            primary_occs = [queues[primary_school].pop()
                            for _ in range(min(self.P, len(queues[primary_school])))]

            other_schools = [s for s in queues if s != primary_school and queues[s]]
            secondary_occs = []
            if other_schools:
                sec_school = self.rng.choice(other_schools)
                secondary_occs = [queues[sec_school].pop()
                                  for _ in range(min(self.S, len(queues[sec_school])))]

            batch_occs = primary_occs + secondary_occs
            if not batch_occs:
                continue
            batch_indices = []
            for poet, chunk in batch_occs:
                batch_indices.extend(chunk)
            yield batch_indices

train_sampler = FullSweepSchoolGroupedSampler(
    train_records, primary_per_batch=3, secondary_per_batch=1,
    poems_per_poet=2, seed=SEED,
)

covered_indices = set()
covered_poets = set()
n_micro_batches = 0
for batch_indices in train_sampler:
    covered_indices.update(batch_indices)
    covered_poets.update(train_records[i]["poet"] for i in batch_indices)
    n_micro_batches += 1

coverage_pct = 100 * len(covered_indices) / len(train_records)

print("تعداد میکروبچ‌ها در یک epoch:", n_micro_batches)
print("تعداد نمونه‌های دیده‌شده در یک epoch:", sum(1 for _ in range(n_micro_batches)) * 8, "(تقریبی)")
print(f"پوششِ رکوردها در یک epoch: {coverage_pct:.2f}٪  (نسخه‌ی قبلی: ~0.34٪)")
print("شاعرانِ پوشش‌داده‌شده:", len(covered_poets), "/ 150")

assert len(covered_poets) == 150, "❌ برخی شاعران در این epoch اصلاً دیده نشدند!"
assert coverage_pct > 95, f"❌ پوششِ دیتاست خیلی پایین است: {coverage_pct:.2f}٪"

MICRO_BATCH_SIZE = 8
TARGET_BATCH_SIZE = 256
ACCUM_STEPS = TARGET_BATCH_SIZE // MICRO_BATCH_SIZE

opt_steps_per_epoch = n_micro_batches // ACCUM_STEPS
print(f"\nACCUM_STEPS={ACCUM_STEPS} | میکروبچ در هر epoch={n_micro_batches}")
print(f"گام‌های بهینه‌سازیِ واقعی در هر epoch: ~{opt_steps_per_epoch}  (نسخه‌ی قبلی: ~1.2)")
print(f"برای رسیدن به ~3000 گامِ واقعی، فقط ~{3000 // max(opt_steps_per_epoch,1)} epoch کافی است "
      f"(نسخه‌ی قبلی برایِ همین تعداد گام به ~2500 epoch نیاز داشت)")

In [ ]:


import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5EncoderModel, ByT5Tokenizer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

MODEL_NAME = "google/byt5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def compute_d_struct(meta_v, meta_s, meta_q, alpha=0.0, beta=0.0, gamma=0.0):
    def match_indicator(x):
        eq = x.unsqueeze(0) == x.unsqueeze(1)
        unknown = (x.unsqueeze(0) == -1) | (x.unsqueeze(1) == -1)
        return (eq & ~unknown).float()
    Iv = match_indicator(meta_v); Is = match_indicator(meta_s); Iq = match_indicator(meta_q)
    return 1 - (alpha * Iv + beta * Is + gamma * Iq)

def compute_w(d_struct, tau_w=0.5):
    return torch.exp((1 - d_struct) / tau_w)

def ld_supcon_loss(z, labels, meta_v, meta_s, meta_q,
                    tau=0.07, tau_w=0.5, alpha=0.0, beta=0.0, gamma=0.0, eps=1e-8):
    B = z.shape[0]
    device = z.device
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.t()) / tau
    d_struct = compute_d_struct(meta_v, meta_s, meta_q, alpha, beta, gamma)
    w = compute_w(d_struct, tau_w)
    labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
    self_mask = torch.eye(B, dtype=torch.bool, device=device)
    pos_mask = labels_eq & ~self_mask
    neg_mask = ~labels_eq
    exp_sim = torch.exp(sim)
    neg_sum = (w * exp_sim * neg_mask.float()).sum(dim=1)
    log_denom = torch.log(exp_sim + neg_sum.unsqueeze(1) + eps)
    log_prob = sim - log_denom
    pos_count = pos_mask.sum(dim=1)
    loss_per_anchor = -(log_prob * pos_mask.float()).sum(dim=1) / (pos_count.float() + eps)
    valid = pos_count > 0
    return loss_per_anchor[valid].mean()

class ByT5PACLEncoder(nn.Module):
    def __init__(self, model_name=MODEL_NAME, use_grad_checkpointing=True):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        if use_grad_checkpointing:
            self.encoder.gradient_checkpointing_enable()

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-8)
        z = F.normalize(pooled, dim=1)
        return z

tokenizer = ByT5Tokenizer.from_pretrained(MODEL_NAME)
model = ByT5PACLEncoder().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"مدل بارگذاری شد: {MODEL_NAME} | تعداد پارامترها: {n_params:,}")
assert n_params > 4e8, f"⚠️ تعداد پارامتر ({n_params:,}) به byt5-base نمی‌خورد"

LR = 3e-5
TARGET_BATCH_SIZE = 256
MICRO_BATCH_SIZE = 8
ACCUM_STEPS = TARGET_BATCH_SIZE // MICRO_BATCH_SIZE

NUM_EPOCHS = 15
STOP_AFTER_EPOCH = 9
TOTAL_OPT_STEPS = opt_steps_per_epoch * NUM_EPOCHS
print(f"سقفِ epoch (برایِ زمان‌بندیِ LR): {NUM_EPOCHS} | توقفِ واقعی: epoch {STOP_AFTER_EPOCH} | بودجهِ گام‌ها: {TOTAL_OPT_STEPS}")

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_OPT_STEPS)

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available() and not USE_BF16)
print(f"AMP dtype برایِ آموزش: {AMP_DTYPE}")

print("✅ مدل، Vanilla-SupCon loss (α=β=γ=0)، optimizer/scheduler آماده‌اند.")


In [ ]:


import os
import time
import itertools
from tqdm.auto import tqdm

from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build('drive', 'v3')

def purge_old_checkpoint_versions(filename):
    query = f"name='{filename}' and trashed=true"
    results = drive_service.files().list(q=query, fields="files(id, name, size)").execute()
    files = results.get("files", [])
    for f in files:
        drive_service.files().delete(fileId=f["id"]).execute()
    if files:
        total_gb = sum(int(f.get("size", 0)) for f in files) / 1e9
        print(f"\U0001f5d1\ufe0f {len(files)} نسخه‌ی قدیمیِ '{filename}' دائماً حذف شد (~{total_gb:.2f} گیگ آزاد شد)")

model.encoder.gradient_checkpointing_disable()
print("✅ gradient checkpointing خاموش شد")

PRIMARY_PER_BATCH = 6
SECONDARY_PER_BATCH = 2
POEMS_PER_POET = 2
MICRO_BATCH_SIZE = (PRIMARY_PER_BATCH + SECONDARY_PER_BATCH) * POEMS_PER_POET
TARGET_BATCH_SIZE = 256
ACCUM_STEPS = TARGET_BATCH_SIZE // MICRO_BATCH_SIZE
print(f"MICRO_BATCH_SIZE={MICRO_BATCH_SIZE} | ACCUM_STEPS={ACCUM_STEPS}")

n_batches_estimate = sum(1 for _ in FullSweepSchoolGroupedSampler(
    train_records, primary_per_batch=PRIMARY_PER_BATCH, secondary_per_batch=SECONDARY_PER_BATCH,
    poems_per_poet=POEMS_PER_POET, seed=0))
opt_steps_per_epoch_actual = n_batches_estimate // ACCUM_STEPS

print(f"شمارشِ واقعی: n_batches={n_batches_estimate} | n//ACCUM_STEPS={opt_steps_per_epoch_actual} "
      f"| مقدارِ قبلی: {opt_steps_per_epoch}")

if opt_steps_per_epoch_actual != opt_steps_per_epoch:
    print(f"\u26a0\ufe0f عدمِ تطابق ({opt_steps_per_epoch} \u2192 {opt_steps_per_epoch_actual})\u061b TOTAL_OPT_STEPS بازسازی می‌شود.")
    opt_steps_per_epoch = opt_steps_per_epoch_actual
else:
    print("✅ عدد جدید با قبلی برابر است.")

TOTAL_OPT_STEPS = opt_steps_per_epoch * NUM_EPOCHS
print(f"TOTAL_OPT_STEPS نهایی: {TOTAL_OPT_STEPS}")

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available() and not USE_BF16)

def clean(v):
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

def make_tensors(batch_records):
    labels = torch.tensor([poet_to_id[r["poet"]] for r in batch_records], dtype=torch.long)
    meta_v = torch.tensor([rhythm_to_id.get(clean(r.get("rhythm")), -1) for r in batch_records], dtype=torch.long)
    meta_s = torch.tensor([school_to_id.get(clean(r.get("school")), -1) for r in batch_records], dtype=torch.long)
    meta_q = torch.tensor([format_to_id.get(clean(r.get("format")), -1) for r in batch_records], dtype=torch.long)
    texts  = [r["text"] for r in batch_records]
    return texts, labels, meta_v, meta_s, meta_q

def forward_loss(batch_records):
    texts, labels, meta_v, meta_s, meta_q = make_tensors(batch_records)
    enc = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    labels, meta_v, meta_s, meta_q = labels.to(DEVICE), meta_v.to(DEVICE), meta_s.to(DEVICE), meta_q.to(DEVICE)
    with torch.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu",
                         dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
        z = model(enc["input_ids"], enc["attention_mask"])
        loss = ld_supcon_loss(z, labels, meta_v, meta_s, meta_q)
    return loss

@torch.no_grad()
def run_full_validation(seed=999):
    model.eval()
    val_sampler = FullSweepSchoolGroupedSampler(
        val_records, primary_per_batch=PRIMARY_PER_BATCH, secondary_per_batch=SECONDARY_PER_BATCH,
        poems_per_poet=POEMS_PER_POET, seed=seed,
    )
    total_loss, n_batches = 0.0, 0
    for batch_indices in val_sampler:
        loss = forward_loss([val_records[i] for i in batch_indices])
        total_loss += loss.item()
        n_batches += 1
    model.train()
    return total_loss / max(n_batches, 1)

def save_checkpoint(path, global_step, epoch, micro_step_in_epoch, best_val_loss, patience_counter, history):
    tmp_path = path + ".tmp"
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "global_step": global_step,
        "epoch": epoch,
        "micro_step_in_epoch": micro_step_in_epoch,
        "best_val_loss": best_val_loss,
        "patience_counter": patience_counter,
        "history": history,
        "model_name": MODEL_NAME,
        "total_opt_steps": TOTAL_OPT_STEPS,
        "ablation": "vanilla_supcon_alpha_beta_gamma_0",
    }, tmp_path)
    os.replace(tmp_path, path)

LOCAL_CKPT_DIR = "/content/local_checkpoints_vanilla"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
LOCAL_CKPT_LAST = f"{LOCAL_CKPT_DIR}/last_local.pt"

CKPT_EPOCH9_FINAL = f"{CKPT_DIR}/vanilla_byt5base_epoch9_final.pt"
SAVE_EVERY_N_OPT_STEPS = 40

local_ckpt = torch.load(LOCAL_CKPT_LAST, map_location="cpu") if os.path.exists(LOCAL_CKPT_LAST) else None

if local_ckpt:
    ckpt, ckpt_source = local_ckpt, "محلی (Colab runtime)"
else:
    ckpt, ckpt_source = None, None

if ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    global_step = ckpt["global_step"]
    start_epoch = ckpt["epoch"]
    resume_micro_step = ckpt["micro_step_in_epoch"]
    best_val_loss = ckpt["best_val_loss"]
    patience_counter = ckpt["patience_counter"]
    history = ckpt["history"]
    print(f"\U0001f504 Resume از checkpointِ {ckpt_source} (global_step={global_step}): epoch={start_epoch}, "
          f"micro_step_in_epoch={resume_micro_step}, best_val_loss={best_val_loss:.4f}")
else:
    global_step, start_epoch, resume_micro_step = 0, 1, 0
    best_val_loss, patience_counter, history = float("inf"), 0, []
    print("\U0001f195 هیچ checkpointی پیدا نشد؛ شروع از صفر.")

if start_epoch > STOP_AFTER_EPOCH:
    print(f"\u2705 آموزش قبلاً تا epoch {start_epoch-1} پیش رفته که >= {STOP_AFTER_EPOCH} است؛ کاری برای انجام نیست، به سلولِ بعدی (استخراج embedding) برو.")

scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_OPT_STEPS,
                               last_epoch=(global_step - 1) if global_step > 0 else -1)
print(f"scheduler بازسازی شد: T_max={TOTAL_OPT_STEPS} | last_epoch={global_step - 1 if global_step > 0 else -1}")

model.train()
optimizer.zero_grad()

for epoch in range(start_epoch, STOP_AFTER_EPOCH + 1):
    t0 = time.time()
    is_resumed_epoch = (epoch == start_epoch and resume_micro_step > 0)
    train_sampler = FullSweepSchoolGroupedSampler(
        train_records, primary_per_batch=PRIMARY_PER_BATCH, secondary_per_batch=SECONDARY_PER_BATCH,
        poems_per_poet=POEMS_PER_POET, seed=SEED * 1000 + epoch,
    )

    batch_iter = enumerate(train_sampler)
    skip_n = resume_micro_step if is_resumed_epoch else 0
    if skip_n > 0:
        batch_iter = itertools.islice(batch_iter, skip_n, None)
        print(f"  \u23ed\ufe0f رد کردنِ {skip_n} میکروبچِ قبلاً پردازش‌شده")

    running_loss, n_micro = 0.0, 0
    epoch_opt_steps = 0
    pbar = tqdm(batch_iter, desc=f"epoch {epoch}/{STOP_AFTER_EPOCH}", initial=skip_n, total=n_batches_estimate)

    for micro_step, batch_indices in pbar:
        batch_records = [train_records[i] for i in batch_indices]
        loss = forward_loss(batch_records)
        scaler.scale(loss / ACCUM_STEPS).backward()
        running_loss += loss.item()
        n_micro += 1

        if (micro_step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            epoch_opt_steps += 1
            pbar.set_postfix(opt_step=global_step, avg_loss=f"{running_loss/n_micro:.4f}",
                              lr=f"{scheduler.get_last_lr()[0]:.2e}")

            if global_step % SAVE_EVERY_N_OPT_STEPS == 0:
                save_checkpoint(LOCAL_CKPT_LAST, global_step, epoch, micro_step + 1,
                                 best_val_loss, patience_counter, history)

    train_loss_avg = running_loss / max(n_micro, 1)
    val_loss = run_full_validation()
    elapsed = time.time() - t0
    partial_note = " (\u26a0\ufe0f جزئی — این epoch با resume ادامه یافت)" if is_resumed_epoch else ""

    print(f"epoch={epoch}/{STOP_AFTER_EPOCH} | global_step={global_step} | opt_steps_this_epoch={epoch_opt_steps} "
          f"| train_loss={train_loss_avg:.4f}{partial_note} | val_loss={val_loss:.4f} | {elapsed/60:.1f} دقیقه")
    history.append({"epoch": epoch, "global_step": global_step, "train_loss": train_loss_avg,
                     "val_loss": val_loss, "partial": is_resumed_epoch})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        print(f"  \u2139\ufe0f val_loss بهبود یافت: {best_val_loss:.4f} (فقط برای اطلاع؛ روی توقف اثر ندارد)")
    else:
        print(f"  \u2139\ufe0f val_loss بهبود نیافت (فقط برای اطلاع؛ Early-Stopping در این نوت‌بوک فعال نیست)")

    save_checkpoint(LOCAL_CKPT_LAST, global_step, epoch + 1, 0, best_val_loss, patience_counter, history)

    if epoch == STOP_AFTER_EPOCH:
        save_checkpoint(CKPT_EPOCH9_FINAL, global_step, epoch + 1, 0, best_val_loss, patience_counter, history)
        purge_old_checkpoint_versions("vanilla_byt5base_epoch9_final.pt")
        print(f"\n\u2705 به epoch {STOP_AFTER_EPOCH} رسیدیم \u2014 چک‌پوینتِ نهایی ذخیره شد: {CKPT_EPOCH9_FINAL}")

if os.path.exists(LOCAL_CKPT_LAST):
    os.remove(LOCAL_CKPT_LAST)

print(f"\n\u2705 پایانِ آموزشِ Vanilla-SupCon تا epoch {STOP_AFTER_EPOCH}. چک‌پوینتِ نهایی: {CKPT_EPOCH9_FINAL}")


In [ ]:
test_closed_records = load_jsonl(f"{PHASE3_DIR}/test_closed.jsonl")
test_open_records   = load_jsonl(f"{PHASE3_DIR}/test_open.jsonl")

In [ ]:

import torch, numpy as np
from tqdm.auto import tqdm

ckpt = torch.load(CKPT_EPOCH9_FINAL, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"✅ چک‌پوینتِ Vanilla-SupCon بارگذاری شد: epoch={ckpt['epoch']-1}, "
      f"global_step={ckpt['global_step']}, best_val_loss={ckpt['best_val_loss']:.4f}")

@torch.no_grad()
def encode_texts_fp32_safe(texts, start_batch_size=192, min_batch_size=16):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    embs = np.zeros((len(texts), 1536), dtype=np.float32)
    n = len(order)
    pbar = tqdm(total=n, desc="Encoding")
    start = 0
    batch_size = start_batch_size
    while start < n:
        end = min(start + batch_size, n)
        batch_idx = order[start:end]
        batch = [texts[i] for i in batch_idx]
        try:
            enc = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
            z = model(enc["input_ids"], enc["attention_mask"])
            embs[batch_idx] = z.float().cpu().numpy()
            start = end
            pbar.update(len(batch_idx))
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" not in str(e).lower() and not isinstance(e, torch.cuda.OutOfMemoryError):
                raise
            torch.cuda.empty_cache()
            if batch_size <= min_batch_size:
                raise
            batch_size = max(batch_size // 2, min_batch_size)
            print(f"  \u26a0\ufe0f OOM رخ داد — batch_size به {batch_size} کاهش یافت، تلاشِ دوباره روی همون بخش...")
    pbar.close()
    return embs

def encode_and_save(name, records):
    emb = encode_texts_fp32_safe([r["text"] for r in records], start_batch_size=192)
    assert np.isnan(emb).any(axis=1).sum() == 0, f"\u274c NaN در {name} embeddings!"
    np.save(f"{PHASE3_DIR}/{name}_embeddings_vanilla_supcon.npy", emb)
    print(f"\u2705 {name}:", emb.shape, "ذخیره شد (vanilla-supcon).")
    return emb

train_embeddings       = encode_and_save("train", train_records)
test_closed_embeddings = encode_and_save("test_closed", test_closed_records)
test_open_embeddings   = encode_and_save("test_open", test_open_records)


In [ ]:
from collections import defaultdict

def group_by_k(records, embeddings):
    by_k = defaultdict(lambda: {"records": [], "embeddings": []})
    for r, e in zip(records, embeddings):
        by_k[r["k"]]["records"].append(r)
        by_k[r["k"]]["embeddings"].append(e)
    for k in by_k:
        by_k[k]["embeddings"] = np.stack(by_k[k]["embeddings"])
    return dict(sorted(by_k.items()))

test_closed_by_k = group_by_k(test_closed_records, test_closed_embeddings)
test_open_by_k = group_by_k(test_open_records, test_open_embeddings)

print("=== Test-Closed بر اساسِ k (Vanilla-SupCon) ===")
for k, d in test_closed_by_k.items():
    print(f"  k={k}: {len(d['records']):,} نمونه")
print("\n=== Test-Open بر اساسِ k (Vanilla-SupCon) ===")
for k, d in test_open_by_k.items():
    print(f"  k={k}: {len(d['records']):,} نمونه")

train_poets = [r["poet"] for r in train_records]
train_poem_ids = [r["poem_id"] for r in train_records]


In [ ]:

import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import normalize as sk_normalize

def evaluate_retrieval(query_vecs, query_poets, query_poem_ids,
                        gallery_vecs, gallery_poets, gallery_poem_ids,
                        ks=(1, 5, 10), exclude_same_poem=False, batch_size=500):
    def l2_normalize(mat):
        if sp.issparse(mat):
            return sk_normalize(mat, norm="l2", axis=1)
        mat = np.asarray(mat, dtype=np.float32)
        return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)

    q = l2_normalize(query_vecs)
    g = l2_normalize(gallery_vecs)
    query_poets = np.asarray(query_poets); gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids); gallery_poem_ids = np.asarray(gallery_poem_ids)

    max_k = max(ks); n_gallery = g.shape[0]
    recall_hits = {k: 0 for k in ks}; mrr_total = 0.0; valid_queries = 0
    n_queries = q.shape[0]

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_q = q[start:end]
        sims = batch_q @ g.T
        if sp.issparse(sims): sims = sims.toarray()
        sims = np.asarray(sims, dtype=np.float32)

        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
            sims = np.where(same_poem_mask, -np.inf, sims)

        k_eff = min(max_k, n_gallery - 1) if exclude_same_poem else min(max_k, n_gallery)
        top_idx = np.argpartition(-sims, kth=k_eff - 1, axis=1)[:, :k_eff]
        row_idx = np.arange(top_idx.shape[0])[:, None]
        top_sims = sims[row_idx, top_idx]
        order = np.argsort(-top_sims, axis=1)
        top_idx_sorted = top_idx[row_idx, order]
        top_sims_sorted = top_sims[row_idx, order]

        batch_poets = query_poets[start:end]
        for i in range(end - start):
            if np.all(np.isneginf(top_sims_sorted[i])): continue
            valid_queries += 1
            retrieved_poets = gallery_poets[top_idx_sorted[i]]
            hit_positions = np.where(retrieved_poets == batch_poets[i])[0]
            if len(hit_positions) > 0:
                rank = hit_positions[0] + 1
                mrr_total += 1.0 / rank
                for k in ks:
                    if rank <= k: recall_hits[k] += 1

    out = {"n_queries": valid_queries, "mrr": mrr_total / valid_queries if valid_queries else float("nan")}
    for k in ks: out[f"recall@{k}"] = recall_hits[k] / valid_queries if valid_queries else float("nan")
    return out

results = {}
print("=== Closed-Set (Gallery = کلِ train) — Vanilla-SupCon ===")
for k in [1, 2, 4, 6]:
    if k not in test_closed_by_k: continue
    d = test_closed_by_k[k]
    res = evaluate_retrieval(
        query_vecs=d["embeddings"], query_poets=[r["poet"] for r in d["records"]],
        query_poem_ids=[r["poem_id"] for r in d["records"]],
        gallery_vecs=train_embeddings, gallery_poets=train_poets, gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10), exclude_same_poem=False)
    results[("closed", k)] = res
    print(f"  k={k}: {res}")

print("\n=== Zero-Shot/Open (Gallery = leave-one-poem-out از خودِ test_open) — Vanilla-SupCon ===")
for k in [1, 2, 4, 6]:
    if k not in test_open_by_k: continue
    d = test_open_by_k[k]
    res = evaluate_retrieval(
        query_vecs=d["embeddings"], query_poets=[r["poet"] for r in d["records"]],
        query_poem_ids=[r["poem_id"] for r in d["records"]],
        gallery_vecs=d["embeddings"], gallery_poets=[r["poet"] for r in d["records"]],
        gallery_poem_ids=[r["poem_id"] for r in d["records"]],
        ks=(1, 5, 10), exclude_same_poem=True)
    results[("open", k)] = res
    print(f"  k={k}: {res}")